#### Importing Dependencies

In [19]:
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()

True

#### Cmd Execution Function

In [2]:
import subprocess

def run(cmd: str) -> str:
    print(f"  [CMD] {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"  [ERROR] exit={result.returncode} | {result.stderr.strip()}")
    return result.stdout


In [ ]:
response = run("agent-browser open youtube.com --headed")
print(response)

✓ YouTube
  https://www.youtube.com/



#### Tools Creation

In [34]:
@tool(
    "snapshot",
    description="Return interactive accessibility snapshot from agent-browser.")
def snapshot() -> str:
    result = run("agent-browser snapshot -i") 
    return result
    

@tool(
    "navigate",
    description=(
        "Control browser navigation using agent-browser commands. "
        "Supported commands:"
        "- 'agent-browser open <url>' → Navigate to a URL (aliases: goto, navigate)"
        "- 'agent-browser tab new <url>' -> Create New tab (url is optional)"
        "- 'agent-browser back' → Go to previous page"
        "- 'agent-browser forward' → Go to next page"
        "- 'agent-browser reload' → Reload the current page"
        "Example: agent-browser open https://youtube.com"
        "Example: agent-browser tab new https://youtube.com"
    )
)
def navigate(cmd: str) -> str:
    result = run(cmd)
    return result

@tool(
    "interact",
    description=(
        "Interact with webpage elements using agent-browser commands. "
        "Mouse actions: 'agent-browser click <selector>' to click (--new-tab to open in new tab), "
        "'agent-browser dblclick <selector>' to double-click, "
        "'agent-browser hover <selector>' to hover, "
        "'agent-browser drag <source_selector> <target_selector>' to drag and drop. "
        "Text input: 'agent-browser fill <selector> <text>' to clear and fill input, "
        "'agent-browser type <selector> <text>' to type into element, "
        "'agent-browser keyboard type <text>' to type at current focus, "
        "'agent-browser keyboard inserttext <text>' to insert text without key events, "
        "'agent-browser upload <selector> <file_paths>' to upload files. "
        "Keyboard actions: 'agent-browser press <key>' to press a key (Enter, Tab, Control+a), "
        "'agent-browser keydown <key>' to hold key down, "
        "'agent-browser keyup <key>' to release key. "
        "Form controls: 'agent-browser focus <selector>' to focus element, "
        "'agent-browser select <selector> <value>' to select dropdown option, "
        "'agent-browser check <selector>' to check checkbox, "
        "'agent-browser uncheck <selector>' to uncheck checkbox. "
        "Scrolling: 'agent-browser scroll <direction> [pixels]' to scroll (up/down/left/right, optional --selector), "
        "'agent-browser scrollintoview <selector>' to scroll element into view. "
        "Selectors must use the ref format: @e4, @e12, etc. (NOT [ref=e4]). "
        "Example: agent-browser click @e4"
    )
)
def interact(cmd: str) -> str:
    result = run(cmd)
    return result


#### Memory

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

SYSTEM_PROMPT = """You are a helpful browser automation agent. 
You can navigate websites, interact with elements, and take snapshots.
Always take a snapshot first to understand the current page state before interacting.
When the whole task given by user completes just give response as Task Completed."""

class ConversationMemory:
    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.history = [SystemMessage(content=system_prompt)]

    def add(self, role: str, content: str):
        if role == "user":
            self.history.append(HumanMessage(content=content))
        else:
            self.history.append(AIMessage(content=content))

    def get(self):
        return self.history

    def clear(self):
        self.history = [self.history[0]]

memory = ConversationMemory()


#### Agent Implementation

In [35]:
agent = create_agent("groq:openai/gpt-oss-120b", tools=[snapshot,navigate,interact])

In [36]:
def chat(user_input: str) -> str:
    memory.add("user", user_input)
    print(f"\n[User] {user_input}")
    print("-" * 50)

    ai_reply = ""
    for step in agent.stream({"messages": memory.get()}, stream_mode="updates"):
        for node, update in step.items():
            for msg in update.get("messages", []):
                # Tool call decision
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"[Tool Call] {tc['name']}({tc['args']})")
                # Tool result
                elif hasattr(msg, "name") and msg.name:
                    preview = (msg.content or "")[:300].replace("\n", " ")
                    print(f"[Tool Result] {msg.name} → {preview}")
                # Final AI reply
                elif hasattr(msg, "content") and msg.content:
                    ai_reply = msg.content
                    print(f"[Agent] {ai_reply}")

    print("-" * 50)
    memory.add("ai", ai_reply)
    return ai_reply



In [ ]:
chat("")